# 02 — Data Cleaning, Boundary Filtering & Integrity Verification

**Objective:**
1. Audit and filter raw OSM bounding-box coordinates strictly inside Karnataka (excluding neighboring Telangana, Tamil Nadu, and Maharashtra records).
2. Standardize ChargeWise station attributes while **preserving missing values** (`opening_hours`, `max_power_kw`, `pricing_verified`) without fabrication.
3. Parse VAHAN monthly date strings (`DD-MMM-YY`) cleanly, compute EV penetration percentage (`ev_penetration_pct`), and export clean analytical CSVs to `data/processed/`.

In [1]:
import json
from pathlib import Path
import numpy as np
import pandas as pd

REPO = Path("..").resolve()
PROC = REPO / "data" / "processed"
PROC.mkdir(parents=True, exist_ok=True)

# 1. Inspect raw OSM bounding-box leakage outside Karnataka
osm_raw = json.loads((REPO / "data" / "raw" / "osm_karnataka_chargers.json").read_text())
osm_rows = []
for el in osm_raw.get("elements", []):
    lat = el.get("lat") or el.get("center", {}).get("lat")
    lon = el.get("lon") or el.get("center", {}).get("lon")
    tags = el.get("tags", {})
    in_blr = 12.75 <= lat <= 13.20 and 77.35 <= lon <= 77.85
    in_south = 12.10 <= lat <= 13.80 and 74.65 <= lon <= 77.35
    in_central = 13.20 <= lat <= 15.80 and 74.30 <= lon <= 77.20
    in_north = 15.50 <= lat <= 17.60 and 74.30 <= lon <= 77.20
    osm_rows.append({
        "osm_id": el["id"],
        "lat": lat,
        "lon": lon,
        "in_karnataka_strict": bool(in_blr or in_south or in_central or in_north),
        "has_operator_tag": bool(tags.get("operator") or tags.get("network") or tags.get("brand")),
        "has_city_tag": bool(tags.get("addr:city")),
    })

osm_audit_df = pd.DataFrame(osm_rows)
print("OSM raw bbox breakdown (Karnataka vs Border Spillover):")
print(osm_audit_df["in_karnataka_strict"].value_counts())
print("OSM missing addr:city rate:", (1 - osm_audit_df["has_city_tag"].mean()) * 100, "%")

OSM raw bbox breakdown (Karnataka vs Border Spillover):
in_karnataka_strict
True     71
False    21
Name: count, dtype: int64
OSM missing addr:city rate: 100.0 %


In [2]:
# 2. Clean ChargeWise Karnataka station export
stations = pd.read_csv(PROC / "karnataka_stations_raw_export.csv")
stations["operator"] = stations["operator"].fillna("Unlisted / Independent").replace("", "Unlisted / Independent")
stations["opening_hours"] = stations["opening_hours"].replace("", np.nan)
stations["max_power_kw"] = pd.to_numeric(stations["max_power_kw"], errors="coerce")
stations["is_verified_tier"] = stations["verification_status"].isin([
    "operational_status_verified", "location_verified", "live_availability_confirmed"
])
stations["region_group"] = np.where(
    stations["district"] == "Bengaluru Urban", "Bengaluru Urban", "Rest of Karnataka / Highways"
)

# Check duplicates on coordinates + name
dupes = stations.duplicated(subset=["name", "latitude", "longitude"]).sum()
print(f"Exact name+coordinate duplicates in clean export: {dupes}")
print("\nMissingness audit (% missing per column):")
print((stations.isna().mean() * 100).round(1))

stations.to_csv(PROC / "karnataka_stations_clean.csv", index=False)

# 3. Clean VAHAN state EV sales time series
ev_dir = REPO / "data" / "raw" / "india_ev_market"
sales = pd.read_csv(ev_dir / "electric_vehicle_sales_by_state.csv")
sales["date_parsed"] = pd.to_datetime(sales["date"], format="%d-%b-%y")
sales["ev_penetration_pct"] = np.where(
    sales["total_vehicles_sold"] > 0,
    (sales["electric_vehicles_sold"] / sales["total_vehicles_sold"]) * 100.0,
    0.0,
)
sales = sales.sort_values(["state", "vehicle_category", "date_parsed"]).reset_index(drop=True)
sales.to_csv(PROC / "india_ev_sales_clean.csv", index=False)

ka_sales = sales[sales["state"] == "Karnataka"].copy()
ka_sales.to_csv(PROC / "karnataka_ev_sales_clean.csv", index=False)
print("Saved clean Karnataka EV sales rows:", len(ka_sales))

Exact name+coordinate duplicates in clean export: 1

Missingness audit (% missing per column):
station_id              0.0
name                    0.0
operator                0.0
city                    0.0
district                0.0
state                   0.0
latitude                0.0
longitude               0.0
charging_type           0.0
opening_hours          94.2
pricing_verified        0.0
verification_status     0.0
confidence_score        0.0
source                  0.0
connector_types         0.0
max_power_kw           80.2
total_connectors        0.0
is_verified_tier        0.0
region_group            0.0
dtype: float64
Saved clean Karnataka EV sales rows: 72
